# Custom CUDA-Optimized GEMM on Tesla T4 (Google Colab)
**NVIDIA DevTech Internship Showcase**

This notebook demonstrates building, benchmarking, and profiling custom CUDA GEMM kernels (Naive vs. Tiled Shared Memory vs. cuBLAS) on an NVIDIA Tesla T4 GPU.

### Step 1: Verify NVIDIA GPU Allocation
Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

### Step 2: Write or Verify `gemm.cu`
If you uploaded `gemm.cu` directly into Colab's file pane (left sidebar), skip this cell. Otherwise, run the cell below to write `gemm.cu` directly.

In [ ]:
!ls -lh gemm.cu

### Step 3: Compile `gemm.cu`
- `-O3`: Maximum host and device optimization
- `-arch=sm_75`: Targets Tesla T4 (Turing architecture)
- `-lcublas`: Links NVIDIA cuBLAS library

In [ ]:
!nvcc -O3 -std=c++14 -arch=sm_75 gemm.cu -o gemm -lcublas
!ls -lh gemm

### Step 4: Run the Benchmark Suite ($N=1024$ and $N=2048$)

In [ ]:
!./gemm

### Step 5: NVIDIA Nsight Compute (`ncu`) Profiling

NVIDIA Nsight Compute is the industry standard kernel profiler used by DevTech engineers.

#### Profiling Command A: High-Level Speed of Light (SOL) Metrics
Measures SM compute throughput % and DRAM bandwidth throughput % for both kernels.

In [ ]:
!ncu --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,dram__throughput.avg.pct_of_peak_sustained_elapsed ./gemm

#### Profiling Command B: Memory Workload Analysis & Bank Conflicts
Inspects shared memory access efficiency and bank conflicts.

In [ ]:
!ncu --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,l1tex__data_pipe_lsu_wavefronts_mem_shared_op_ld.sum --kernel-name gemm_tiled ./gemm

#### Profiling Command C: Export Full Profiler Report
Generates a `.ncu-rep` file that you can download and open in the NVIDIA Nsight Compute GUI on your Mac/PC.

In [ ]:
!ncu --set full -o gemm_profile_report ./gemm